# Lab Instructions

Find a dataset that interests you. I'd recommend starting on [Kaggle](https://www.kaggle.com/). Read through all of the material about the dataset and download a .CSV file.

1. Write a short summary of the data.  Where did it come from?  How was it collected?  What are the features in the data?  Why is this dataset interesting to you?  

2. Identify 5 interesting questions about your data that you can answer using Pandas methods.  

3. Answer those questions!  You may use any method you want (including LLMs) to help you write your code; however, you should use Pandas to find the answers.  LLMs will not always write code in this way without specific instruction.  

4. Write the answer to your question in a text box underneath the code you used to calculate the answer.



## Dataset Summary

**Origin:** The dataset is hosted on Kaggle and was created/uploaded by Debanjan Mondal.

**Collection:** It consists of 6,000 real-world university student queries. It is split into a training set (5,000 samples) and a test set (1,000 samples).

**Features:** The dataset typically contains three main columns:
- **ID:** A unique identifier for the query.
- **Query Text:** The actual text of the question or request submitted by the student (e.g., "When is the exam?", "I cannot access the portal").
- **Priority Label:** A categorical classification of the query's urgency, labeled as High, Medium, or Low.

**Why it is interesting:** This dataset is fascinating because it bridges the gap between Natural Language Processing (NLP) and educational administration. Efficiently sorting student requests is a massive logistical challenge for universities. By analyzing this data, we can build models to automate "triage," ensuring that critical issues (like enrollment errors or payment failures) are addressed before general inquiries (like library hours).

## 5 Interesting Questions

1. **What is the distribution of priority levels?** (Is the dataset balanced, or are "High" priority issues rare?)

2. **Does the length of the query correlate with its priority?** (Are urgent queries shorter and more direct, or longer and more detailed?)

3. **What are the most common keywords in "High" priority queries?** (Do words like "urgent," "fail," or "deadline" appear more often?)

4. **Are there duplicate queries in the dataset?** (Do students ask the exact same questions repeatedly?)

5. **What percentage of queries relate to "exams" or "grades"?** (How much of the support volume is academic vs. administrative?)

## Question 1: What is the distribution of priority levels?

Is the dataset balanced, or are "High" priority issues rare?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset (assuming the file name matches the download)
df = pd.read_csv('student_query_train_5000.csv')

# Calculate the counts of each priority level
priority_counts = df['Priority'].value_counts()

print(priority_counts)

# Optional: Visualize it
priority_counts.plot(kind='bar', title='Distribution of Query Priorities')
plt.show()

**Answer:**

This code calculates the frequency of each label ('High', 'Medium', 'Low').

If the output shows roughly equal numbers (e.g., ~1600 for each), the dataset is balanced, which is ideal for training AI models.

If "Low" is significantly higher than "High" (imbalanced), it reflects real-world help desks where minor issues are more common than emergencies.

## Question 2: Does the length of the query correlate with its priority?

Are urgent queries shorter and more direct, or longer and more detailed?

In [ ]:
# Create a new column for query length
df['Query_Length'] = df['Query'].apply(len)

# Group by Priority and calculate the average length
avg_length_by_priority = df.groupby('Priority')['Query_Length'].mean()

print(avg_length_by_priority)

**Answer:**

This code groups the data by priority and finds the mean character count for each group.

**Hypothesis:** You might find that High priority queries are shorter and more frantic (e.g., "Login failed help!"), while Medium or Low queries are longer and more explanatory.

**Result:** The output provides the specific average number of characters for each category, validating or refuting that hypothesis.

## Question 3: What are the most common keywords in "High" priority queries?

Do words like "urgent," "fail," or "deadline" appear more often?

In [ ]:
from collections import Counter

# Filter for High priority only
high_priority_df = df[df['Priority'] == 'High']

# Join all queries into one massive string and split into words
all_words = ' '.join(high_priority_df['Query']).lower().split()

# Count the 10 most common words
common_words = Counter(all_words).most_common(10)

print(common_words)

**Answer:**

This code isolates the urgent queries and counts word frequency.

**Likely Answer:** You will likely see stop words (the, is, to) at the top, but looking slightly lower in the list will reveal domain-specific triggers.

**Insight:** If words like "payment," "crash," or "deadline" appear frequently here, these are the key features a model should learn to look for.

## Question 4: Are there duplicate queries in the dataset?

Do students ask the exact same questions repeatedly?

In [ ]:
# Check for duplicate rows in the 'Query' column
duplicate_count = df.duplicated(subset=['Query']).sum()

print(f"Number of duplicate queries: {duplicate_count}")

# View a few duplicates to see if they are generic questions
if duplicate_count > 0:
    print(df[df.duplicated(subset=['Query'], keep=False)].head())

**Answer:**

This code counts how many rows contain the exact same text as another row.

**Significance:** In student datasets, duplicates are common for generic questions like "When do classes start?"

**Data Cleaning:** If this number is high, you might need to drop duplicates to prevent your AI model from memorizing frequent sentences rather than learning the underlying patterns.

## Question 5: What percentage of queries relate to "exams" or "grades"?

How much of the support volume is academic vs. administrative?

In [ ]:
# Define a list of keywords to search for
keywords = ['exam', 'grade', 'test', 'score', 'gpa']

# Create a boolean mask where True means the query contains one of the keywords
# The join creates a regex pattern: "exam|grade|test|score|gpa"
keyword_mask = df['Query'].str.contains('|'.join(keywords), case=False, na=False)

# Calculate the percentage
percentage = (keyword_mask.sum() / len(df)) * 100

print(f"Percentage of queries related to exams/grades: {percentage:.2f}%")

**Answer:**

This code searches every query for specific academic terms using a regular expression.

**Context:** This tells you the "topic intent" of the dataset.

**Result:** If the result is, for example, 40%, you know that a significant portion of the support team's workload is purely academic performance inquiries, distinct from technical support or financial aid.